# Práctica 1 – Notebook 2: Predicciones con el Modelo Final

**Asignatura:** Aprendizaje Automático 2025-26  
**Grupo:**
- Pablo García Aparicio
- Miguel Merino Sánchez

**NIA:** 100522190  

Este notebook carga el modelo final entrenado (`modelo_final.joblib`) y lo usa para:
1. Generar predicciones sobre el dataset de competición (`bank_competition.pkl`)
2. Guardar las predicciones en `predicciones.csv`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder

# Semilla reproducibilidad
SEED = 100522190
np.random.seed(SEED)

print('Librerías cargadas.')

## 1. Cargar modelo final

In [ ]:
modelo = joblib.load('modelo_final.joblib')
print(f'Modelo cargado: {type(modelo.named_steps["clf"]).__name__}')
print(modelo)

## 2. Cargar datos de competición y aplicar preprocesamiento de pdays

In [ ]:
df_comp = pd.read_pickle('bank_competition.pkl')
print(f'Dataset competición: {df_comp.shape[0]} instancias, {df_comp.shape[1]} variables')
print('Columnas:', list(df_comp.columns))
df_comp.head()

In [ ]:
# ── Mismo preprocesamiento de pdays que en el notebook 1 ──────────────────
df_comp['pdays_contactado'] = (df_comp['pdays'] != -1).astype(int)
df_comp['pdays'] = df_comp['pdays'].replace(-1, np.nan)

print('Preprocesamiento pdays aplicado.')
print(f"pdays_contactado (0=no contactado, 1=contactado): {df_comp['pdays_contactado'].value_counts().to_dict()}")

## 3. Generar predicciones

In [ ]:
# Predicciones de clase y probabilidades
y_pred = modelo.predict(df_comp)
y_proba = modelo.predict_proba(df_comp)[:, 1]

# Convertir a etiquetas legibles (yes/no)
# Reconstruir el LabelEncoder con el mismo mapping
le = LabelEncoder()
le.classes_ = np.array(['no', 'yes'])  # orden estándar
y_pred_labels = le.inverse_transform(y_pred)

print(f'Predicciones generadas: {len(y_pred_labels)} instancias')
print(f'Distribución: {pd.Series(y_pred_labels).value_counts().to_dict()}')

## 4. Guardar predicciones en CSV

In [ ]:
pred_df = pd.DataFrame({
    'deposit': y_pred_labels,
    'probabilidad_yes': y_proba.round(4)
})

# Guardar solo la columna deposit (sin probabilidades) para la competición
pred_df[['deposit']].to_csv('predicciones.csv', index=False)
print('Predicciones guardadas en: predicciones.csv')

# Preview
print('\nPrimeras 10 predicciones:')
display(pred_df.head(10))

## 5. Verificación: Predicciones del modelo en 2 instancias específicas

*(Estas mismas instancias se comparan después con la app Streamlit para verificar que las predicciones coinciden)*

In [ ]:
# Instancia 1: Primer cliente del dataset de competición
instancia_1 = df_comp.iloc[[0]]
pred_1 = modelo.predict(instancia_1)[0]
proba_1 = modelo.predict_proba(instancia_1)[0, 1]

# Instancia 2: Segundo cliente del dataset de competición
instancia_2 = df_comp.iloc[[1]]
pred_2 = modelo.predict(instancia_2)[0]
proba_2 = modelo.predict_proba(instancia_2)[0, 1]

print('=== Verificación de predicciones ===')
print(f'Instancia 1:')
display(instancia_1)
pred_label_1 = le.inverse_transform([pred_1])[0]
print(f'  → Predicción: {pred_label_1} (Prob. YES={proba_1:.4f})')

print(f'\nInstancia 2:')
display(instancia_2)
pred_label_2 = le.inverse_transform([pred_2])[0]
print(f'  → Predicción: {pred_label_2} (Prob. YES={proba_2:.4f})')

print('\n⚠️  Estas predicciones deben coincidir exactamente con las de la app Streamlit.')